# PariShiksha: experiment log

This notebook records how the assistant was built and measured, stage by stage,
with the original outputs kept as evidence. The reusable, tested implementation
lives in the [`parishiksha/`](../parishiksha) package; use `python -m parishiksha`
to run it.

Requirements: the two NCERT PDFs in `data/raw/` and a `.env` with `GROQ_API_KEY`
and `GEMINI_API_KEY` (see the README).

In [ ]:
# Run from the project root so the relative data/ paths resolve
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Load the chapter PDFs

In [1]:
import fitz 
import re
import json
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CH4_PATH = RAW_DIR / "iesc104.pdf"
CH6_PATH = RAW_DIR / "iesc106.pdf"

print("Setup complete ")
print(f"Ch4 exists: {CH4_PATH.exists()}")
print(f"Ch6 exists: {CH6_PATH.exists()}")

Setup complete 
Ch4 exists: True
Ch6 exists: True


In [3]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text()
        pages.append({
            "page": page_num,
            "text": text.strip()
        })
    doc.close()
    return pages

ch4_pages = extract_text_from_pdf(CH4_PATH)
ch6_pages = extract_text_from_pdf(CH6_PATH)

print(f"Ch4 — Pages extracted: {len(ch4_pages)}")
print(f"Ch6 — Pages extracted: {len(ch6_pages)}")

# Quick sanity check — print first 300 chars of page 2
print("\n── Ch4 Page 2 preview ──")
print(ch4_pages[1]["text"][:300])

print("\n── Ch6 Page 2 preview ──")
print(ch6_pages[1]["text"][:300])

Ch4 — Pages extracted: 24
Ch6 — Pages extracted: 22

── Ch4 Page 2 preview ──
Describing Motion Around Us
49
about some more physical quantities, such as displacement, average 
velocity and average acceleration. You will also learn to describe motion 
not only in words, but also with numbers, equations and graphs.
4.1 Motion in a Straight Line
You have learnt that when an obj

── Ch6 Page 2 preview ──
How Forces Affect Motion
95
Fig. 6.3: (a) Pushing a box 
kept on table or floor
Force by hand
Force of friction
Fig. 6.2: Measuring the 
weight of an object using a 
spring balance
While learning about force earlier, did you notice that whenever any 
type of force acting on an object was described, 


## 2. Label each page by content type
Worked examples, activities and end-of-chapter questions are chunked differently later, so every page gets a label first.

In [4]:
def classify_content_type(text):
    text_lower = text.lower()
    
    if re.search(r'example\s+\d+[\.:]\s*|answer\s*:', text_lower):
        return "worked_example"
    
    if re.search(r'revise,\s*reflect|pause and ponder|journey beyond|at a glance', text_lower):
        return "end_of_chapter"
    
    if re.search(r'activity\s+\d+[\.:]\s*', text_lower):
        return "activity"
    
    return "concept"

def split_into_sections(pages, chapter_name):
    sections = []
    current_section = "Introduction"
    
    for page_data in pages:
        text = page_data["text"]
        page_num = page_data["page"]
        
        # Detect section headings like 4.1, 4.2, 6.1 etc.
        heading_match = re.search(r'\n(\d+\.\d+[\.\d]*\s+[A-Z][^\n]{5,60})', text)
        if heading_match:
            current_section = heading_match.group(1).strip()
        
        content_type = classify_content_type(text)
        
        sections.append({
            "chapter": chapter_name,
            "page": page_num,
            "section": current_section,
            "content_type": content_type,
            "text": text
        })
    
    return sections

ch4_sections = split_into_sections(ch4_pages, "Chapter 4 - Describing Motion Around Us")
ch6_sections = split_into_sections(ch6_pages, "Chapter 6 - How Forces Affect Motion")

all_sections = ch4_sections + ch6_sections

df = pd.DataFrame(all_sections)
print("Content type distribution:")
print(df["content_type"].value_counts())
print(f"\nTotal pages classified: {len(all_sections)}")
print("\nSample section labels (Ch4):")
for s in ch4_sections[:5]:
    print(f"  Page {s['page']} | {s['content_type']:15} | {s['section'][:50]}")

Content type distribution:
content_type
concept           14
activity          13
worked_example    12
end_of_chapter     7
Name: count, dtype: int64

Total pages classified: 46

Sample section labels (Ch4):
  Page 1 | concept         | Introduction
  Page 2 | concept         | 4.1 Motion in a Straight Line
  Page 3 | concept         | 4.1.2 Distance travelled and displacement
  Page 4 | end_of_chapter  | 4.1.2 Distance travelled and displacement
  Page 5 | worked_example  | 4.1.3 Average speed and average velocity


## 3. Tokenizer comparison
How GPT-2 BPE, BERT WordPiece and T5 SentencePiece split physics text and formulas.

In [5]:
from transformers import AutoTokenizer

# Three tokenizer families
gpt2_tokenizer    = AutoTokenizer.from_pretrained("gpt2")
bert_tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
t5_tokenizer      = AutoTokenizer.from_pretrained("t5-small")

# 5 representative passages from our corpus
passages = [
    "Displacement is the net change in the position of an object between the two given instants of time.",
    
    "The average velocity of an object in a time interval is given by v_av = s/t, where s is displacement and t is time interval. The SI unit is metre per second (m/s).",
    
    "The kinematic equations for motion in a straight line with constant acceleration are: v = u + at, s = ut + (1/2)at², and v² = u² + 2as.",
    
    "Newton's second law of motion states: when a net force acts on an object, the object accelerates in the direction of the net force. Mathematically, F = ma.",
    
    "When brakes are applied to a moving vehicle, it moves some distance before coming to a stop. The distance depends upon the velocity, road surface, and braking capacity of the vehicle."
]

results = []
print(f"{'Passage':<6} {'GPT2-BPE':>10} {'BERT-WP':>10} {'T5-SP':>10}  Disagreement Example")
print("-" * 80)

for i, passage in enumerate(passages, 1):
    gpt2_tokens = gpt2_tokenizer.tokenize(passage)
    bert_tokens = bert_tokenizer.tokenize(passage)
    t5_tokens   = t5_tokenizer.tokenize(passage)
    
    g, b, t = len(gpt2_tokens), len(bert_tokens), len(t5_tokens)
    
    sci_words = ["displacement", "acceleration", "kinematic", "velocity", "v²"]
    disagreement = ""
    for word in sci_words:
        if word.lower() in passage.lower():
            gw = [tok for tok in gpt2_tokens if word[:4].lower() in tok.lower()]
            bw = [tok for tok in bert_tokens if word[:4].lower() in tok.lower()]
            if gw or bw:
                disagreement = f"'{word}' → GPT2:{gw[:2]} BERT:{bw[:2]}"
                break
    
    print(f"P{i:<5} {g:>10} {b:>10} {t:>10}  {disagreement[:55]}")
    results.append({"passage": i, "gpt2": g, "bert": b, "t5": t})

print("\n── Detailed token boundaries for Passage 3 (formula-heavy) ──")
p3 = passages[2]
print(f"\nGPT2 tokens: {gpt2_tokenizer.tokenize(p3)}")
print(f"\nBERT tokens: {bert_tokenizer.tokenize(p3)}")
print(f"\nT5   tokens: {t5_tokenizer.tokenize(p3)}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\Ace\OneDrive\Desktop\parishiksha_study_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ace\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

c:\Users\Ace\OneDrive\Desktop\parishiksha_study_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ace\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passage   GPT2-BPE    BERT-WP      T5-SP  Disagreement Example
--------------------------------------------------------------------------------
P1             22         20         22  'displacement' → GPT2:[] BERT:['displacement']
P2             43         44         50  'displacement' → GPT2:['Ġdisplacement'] BERT:['displace
P3             44         44         52  'acceleration' → GPT2:['Ġacceleration'] BERT:['accelera
P4             37         37         40  
P5             36         36         40  'velocity' → GPT2:['Ġvelocity'] BERT:['velocity']

── Detailed token boundaries for Passage 3 (formula-heavy) ──

GPT2 tokens: ['The', 'Ġk', 'inem', 'atic', 'Ġequations', 'Ġfor', 'Ġmotion', 'Ġin', 'Ġa', 'Ġstraight', 'Ġline', 'Ġwith', 'Ġconstant', 'Ġacceleration', 'Ġare', ':', 'Ġv', 'Ġ=', 'Ġu', 'Ġ+', 'Ġat', ',', 'Ġs', 'Ġ=', 'Ġut', 'Ġ+', 'Ġ(', '1', '/', '2', ')', 'at', 'Â²', ',', 'Ġand', 'Ġv', 'Â²', 'Ġ=', 'Ġu', 'Â²', 'Ġ+', 'Ġ2', 'as', '.']

BERT tokens: ['the', 'kin', '##ema', '##tic', 'e

## 4. Wk9 baseline: word-based chunking and BM25

In [6]:
def chunk_sections(sections, chunk_size=300, overlap=50):
    """
    Chunking strategy:
    - concept pages: 300 tokens, 50 overlap
    - worked_example: kept whole (problem + solution must stay together)
    - end_of_chapter: split into individual questions
    - activity: kept whole
    
    Tokenizer: BERT WordPiece (used consistently for both index and query)
    """
    chunks = []
    chunk_id = 0
    words_per_token = 0.75  

    for section in sections:
        text = section["text"]
        ctype = section["content_type"]
        
        # Clean text
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'Chapter-\d+\.indd.*', '', text).strip()
        
        if ctype == "worked_example":
            chunks.append({
                "chunk_id": chunk_id,
                "chapter": section["chapter"],
                "section": section["section"],
                "content_type": ctype,
                "page": section["page"],
                "text": text
            })
            chunk_id += 1

        elif ctype == "end_of_chapter":
            questions = re.split(r'(?=\n?\d+\.\s+[A-Z])', text)
            for q in questions:
                q = q.strip()
                if len(q) > 50:
                    chunks.append({
                        "chunk_id": chunk_id,
                        "chapter": section["chapter"],
                        "section": section["section"],
                        "content_type": ctype,
                        "page": section["page"],
                        "text": q
                    })
                    chunk_id += 1

        else:
            words = text.split()
            step = int(chunk_size * words_per_token)
            window = int((chunk_size + overlap) * words_per_token)
            
            if len(words) <= window:
                chunks.append({
                    "chunk_id": chunk_id,
                    "chapter": section["chapter"],
                    "section": section["section"],
                    "content_type": ctype,
                    "page": section["page"],
                    "text": text
                })
                chunk_id += 1
            else:
                for start in range(0, len(words), step):
                    chunk_text = " ".join(words[start:start + window])
                    if len(chunk_text) > 100:
                        chunks.append({
                            "chunk_id": chunk_id,
                            "chapter": section["chapter"],
                            "section": section["section"],
                            "content_type": ctype,
                            "page": section["page"],
                            "text": chunk_text
                        })
                        chunk_id += 1

    return chunks

chunks = chunk_sections(all_sections, chunk_size=300, overlap=50)

# Save to disk, find it in the processed folder 
with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

# Summary
chunks_df = pd.DataFrame(chunks)
print(f"Total chunks created: {len(chunks)}")
print(f"\nChunks by content type:")
print(chunks_df["content_type"].value_counts())
print(f"\nChunks by chapter:")
print(chunks_df["chapter"].value_counts())
print(f"\nAvg chunk length (words): {chunks_df['text'].apply(lambda x: len(x.split())).mean():.0f}")
print(f"\nSample chunk (worked_example):")
ex = chunks_df[chunks_df.content_type == "worked_example"].iloc[0]
print(f"  Section: {ex['section']}")
print(f"  Preview: {ex['text'][:200]}")

Total chunks created: 101

Chunks by content type:
content_type
activity          36
concept           32
end_of_chapter    21
worked_example    12
Name: count, dtype: int64

Chunks by chapter:
chapter
Chapter 6 - How Forces Affect Motion       51
Chapter 4 - Describing Motion Around Us    50
Name: count, dtype: int64

Avg chunk length (words): 225

Sample chunk (worked_example):
  Section: 4.1.3 Average speed and average velocity
  Preview: Exploration|Grade 9 52 Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But how can you describe how f


In [7]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def clean_chunk_text(text):
    text = re.sub(r'\d*\s*Exploration\|Grade\s*9\s*\d*', '', text)
    text = re.sub(r'Fig\.?\s*\d+\.\d+[a-z]?\s*:?[^\n]*', '', text)
    text = re.sub(r'Chapter-\d+\.indd.*', '', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'Table\s+\d+\.\d+\s*:?[^\n]*', '', text)
    text = re.sub(r'Grade\s+\d+\s*Curiosity\s*Chapter\s*\d+', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# Clean chunks
for chunk in chunks:
    chunk["text"] = clean_chunk_text(chunk["text"])

chunks = [c for c in chunks if len(c["text"].split()) > 30]
print(f"Chunks after cleaning: {len(chunks)}")

# Save
with open(PROCESSED_DIR / "chunks.json", "w") as f:
    json.dump(chunks, f, indent=2)

# Build BM25
corpus_tokens = [simple_tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(corpus_tokens)
print("BM25 index rebuilt on clean chunks ✓")

# Define retrieve
def retrieve(query, top_k=3, boost_concept=True):
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    
    if boost_concept:
        for i, chunk in enumerate(chunks):
            if chunk["content_type"] == "concept":
                scores[i] *= 1.3
            elif chunk["content_type"] == "worked_example":
                scores[i] *= 1.2
            elif chunk["content_type"] == "end_of_chapter":
                scores[i] *= 0.8
    
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["bm25_score"] = round(scores[idx], 4)
        results.append(chunk)
    return results

# Test
test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    results = retrieve(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  Rank {i} | Score: {r['bm25_score']} | {r['content_type']} | Page {r['page']}")
        print(f"  Section: {r['section']}")
        print(f"  Preview: {r['text'][:150]}...")

Chunks after cleaning: 71
BM25 index rebuilt on clean chunks ✓

Query: What is displacement and how is it different from distance?

  Rank 1 | Score: 10.7321 | concept | Page 23
  Section: 4.4 Motion in a Plane
  Preview: by drawing a velocity-time graph for its motion. 15. Two cars A and B start moving with a constant acceleration from rest in a straight line. Car A at...

  Rank 2 | Score: 9.8869 | concept | Page 16
  Section: 4.3 Kinematic Equations for Motion in a Straight
  Preview: Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the area enclosed by the velocity...

  Rank 3 | Score: 9.3237 | worked_example | Page 5
  Section: 4.1.3 Average speed and average velocity
  Preview: Motion, i.e., a change in the position of an object, can be described in terms of the total distance travelled by the object and its displacement. But...

Query: State Newton's second law of motion with formula

  Rank 1 | Score: 10.8409 

## 5. Wk9 grounded generation (Llama-3.1-8B on Groq)

In [10]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

GROUNDING_PROMPT = """You are a science study assistant for Class 9 students using NCERT textbooks.

You will be given CONTEXT extracted from NCERT Class 9 Science chapters.
Your job is to answer the student's question STRICTLY using only the provided context.

STRICT RULES:
1. If the answer is present in the context, answer clearly and simply for a Class 9 student.
2. If the answer is NOT in the context, respond with exactly: "I'm sorry, this topic is not covered in the provided chapter content. Please refer to your teacher or other chapters."
3. Do NOT use any outside knowledge, even if you know the answer.
4. Do NOT make up formulas, definitions, or examples not present in the context.
5. Keep answers concise — 3 to 5 sentences maximum.
6. If a formula is in the context, include it in your answer.

CONTEXT:
{context}

STUDENT QUESTION:
{question}

ANSWER:"""

def format_context(retrieved_chunks):
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        parts.append(
            f"[Source {i} | {chunk['chapter']} | {chunk['section']} | Page {chunk['page']}]\n"
            f"{chunk['text']}"
        )
    return "\n\n".join(parts)

def answer(question, top_k=3, temperature=0.0):
    retrieved = retrieve(question, top_k=top_k)
    context = format_context(retrieved)
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=1024
    )
    
    return {
        "question": question,
        "answer": response.choices[0].message.content.strip(),
        "retrieved_chunks": retrieved,
        "context_used": context
    }

# ── Smoke test ────────────────────────────────────────────────────────
test_q = "What is Newton's second law of motion?"
result = answer(test_q)
print(f"Question: {result['question']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved from:")
for c in result['retrieved_chunks']:
    print(f"  - {c['section']} | Page {c['page']} | Score {c['bm25_score']}")

Question: What is Newton's second law of motion?

Answer:
Newton's second law of motion states that the rate of change of momentum of an object is proportional to the net force and takes place in the direction in which the net force acts.

Retrieved from:
  - 6.5 Newton’s Second Law of Motion | Page 11 | Score 12.2819
  - 6.7 Forces Acting on a System of Objects | Page 18 | Score 11.096
  - 6.1 The Concept of Force | Page 1 | Score 10.2997


## 6. Wk9 evaluation (18 questions)

In [11]:
evaluation_questions = [
    # Taken from the textbook (10 questions)
    {
        "id": 1,
        "question": "What is displacement?",
        "expected_topic": "displacement definition",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 2,
        "question": "What are the three kinematic equations of motion?",
        "expected_topic": "kinematic equations v=u+at",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 3,
        "question": "What is uniform circular motion?",
        "expected_topic": "circular motion constant speed",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 4,
        "question": "What is average acceleration?",
        "expected_topic": "average acceleration change in velocity",
        "category": "direct",
        "chapter": "Ch4"
    },
    {
        "id": 5,
        "question": "State Newton's first law of motion.",
        "expected_topic": "object at rest remains at rest",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 6,
        "question": "State Newton's second law of motion.",
        "expected_topic": "F=ma net force acceleration",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 7,
        "question": "State Newton's third law of motion.",
        "expected_topic": "equal and opposite force",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 8,
        "question": "What is the SI unit of force and how is it defined?",
        "expected_topic": "newton 1kg 1ms-2",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 9,
        "question": "What is the force of friction and in which direction does it act?",
        "expected_topic": "friction opposite direction of motion",
        "category": "direct",
        "chapter": "Ch6"
    },
    {
        "id": 10,
        "question": "How do you calculate average speed?",
        "expected_topic": "total distance divided by time",
        "category": "direct",
        "chapter": "Ch4"
    },

    {
        "id": 11,
        "question": "Why does a fielder pull hands back while catching a fast cricket ball?",
        "expected_topic": "Newton second law time force injury",
        "category": "paraphrased",
        "chapter": "Ch6"
    },
    {
        "id": 12,
        "question": "If an object moves in a circle at constant speed is it accelerating?",
        "expected_topic": "direction changes so acceleration exists",
        "category": "paraphrased",
        "chapter": "Ch4"
    },
    {
        "id": 13,
        "question": "What decides how far a car travels after brakes are applied?",
        "expected_topic": "velocity road surface braking capacity",
        "category": "paraphrased",
        "chapter": "Ch4"
    },

    {
        "id": 14,
        "question": "What is photosynthesis?",
        "expected_topic": "REFUSAL - biology not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 15,
        "question": "Explain the water cycle.",
        "expected_topic": "REFUSAL - geography not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 16,
        "question": "What is quantum entanglement?",
        "expected_topic": "REFUSAL - advanced physics not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 17,
        "question": "Who is the Prime Minister of India?",
        "expected_topic": "REFUSAL - not science content",
        "category": "out_of_scope",
        "chapter": "None"
    },
    {
        "id": 18,
        "question": "Explain Newton's law of gravitation and calculate gravitational force between two planets.",
        "expected_topic": "REFUSAL - gravitation chapter not in corpus",
        "category": "out_of_scope",
        "chapter": "None"
    },
]

print(f"Total evaluation questions: {len(evaluation_questions)}")
print(f"Direct:       {sum(1 for q in evaluation_questions if q['category'] == 'direct')}")
print(f"Paraphrased:  {sum(1 for q in evaluation_questions if q['category'] == 'paraphrased')}")
print(f"Out of scope: {sum(1 for q in evaluation_questions if q['category'] == 'out_of_scope')}")

Total evaluation questions: 18
Direct:       10
Paraphrased:  3
Out of scope: 5


In [14]:
import time

def evaluate_answer(question_data, result):
    """
    Manual scoring guide — you review each answer after running.
    Returns a dict with placeholders you fill in.
    """
    return {
        "id": question_data["id"],
        "question": question_data["question"],
        "category": question_data["category"],
        "chapter": question_data["chapter"],
        "expected_topic": question_data["expected_topic"],
        "answer": result["answer"],
        "top_chunk_section": result["retrieved_chunks"][0]["section"],
        "top_chunk_page": result["retrieved_chunks"][0]["page"],
        "top_chunk_type": result["retrieved_chunks"][0]["content_type"],
        "correctness": "?",       # yes / partial / no
        "grounded": "?",          # yes / no
        "refusal_appropriate": "?" if question_data["category"] != "out_of_scope" else "?"
    }

# Run every question
print("Running evaluation... (this may take ~1 min due to API calls)\n")
eval_results = []

for q in evaluation_questions:
    try:
        result = answer(q["question"], top_k=3, temperature=0.0)
        row = evaluate_answer(q, result)
        eval_results.append(row)
        
        print(f"[Q{q['id']:02d}] [{q['category']:12}] {q['question'][:55]}")
        print(f"       Answer: {result['answer'][:120]}...")
        print(f"       Top chunk: {result['retrieved_chunks'][0]['section'][:50]}")
        print()
        
        time.sleep(2)  
        
    except Exception as e:
        print(f"[Q{q['id']:02d}] ERROR: {e}")
        eval_results.append({
            "id": q["id"], "question": q["question"],
            "category": q["category"], "answer": f"ERROR: {e}",
            "correctness": "no", "grounded": "no"
        })

print(f"\nEvaluation complete. {len(eval_results)} questions processed.")

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv("evaluation/evaluation_results_raw.csv", index=False)
print("Saved to evaluation/evaluation_results_raw.csv")

Running evaluation... (this may take ~1 min due to API calls)

[Q01] [direct      ] What is displacement?
       Answer: Displacement is the change in the position of an object from its initial position to its final position. It is a vector ...
       Top chunk: 4.4 Motion in a Plane

[Q02] [direct      ] What are the three kinematic equations of motion?
       Answer: The three kinematic equations of motion are:

1. at = v - u 
2. v = u + at 
3. v^2 = u^2 + 2as...
       Top chunk: 4.3 Kinematic Equations for Motion in a Straight

[Q03] [direct      ] What is uniform circular motion?
       Answer: When an object moves in a circular path with constant (uniform) speed, its motion is called uniform circular motion....
       Top chunk: 4.4 Motion in a Plane

[Q04] [direct      ] What is average acceleration?
       Answer: The average acceleration of an object over a time interval is the change in its velocity divided by the time interval. T...
       Top chunk: 4.1.4 Average accelerati

In [15]:
eval_df = pd.read_csv("evaluation/evaluation_results_raw.csv")
for _, row in eval_df.iterrows():
    print(f"{'─'*60}")
    print(f"Q{row['id']} | {row['category']} | {row['question']}")
    print(f"ANSWER: {row['answer']}")
    print(f"CHUNK: {row['top_chunk_section']}")
    print()

────────────────────────────────────────────────────────────
Q1 | direct | What is displacement?
ANSWER: Displacement is the change in the position of an object from its initial position to its final position. It is a vector quantity, which means it has both magnitude and direction.
CHUNK: 4.4 Motion in a Plane

────────────────────────────────────────────────────────────
Q2 | direct | What are the three kinematic equations of motion?
ANSWER: The three kinematic equations of motion are:

1. at = v - u 
2. v = u + at 
3. v^2 = u^2 + 2as
CHUNK: 4.3 Kinematic Equations for Motion in a Straight

────────────────────────────────────────────────────────────
Q3 | direct | What is uniform circular motion?
ANSWER: When an object moves in a circular path with constant (uniform) speed, its motion is called uniform circular motion.
CHUNK: 4.4 Motion in a Plane

────────────────────────────────────────────────────────────
Q4 | direct | What is average acceleration?
ANSWER: The average acceleration 

In [16]:
scores = {
    1:  ("partial", "yes", "n/a"),
    2:  ("partial", "yes", "n/a"),
    3:  ("yes", "yes", "n/a"),
    4:  ("yes", "yes", "n/a"),
    5:  ("yes", "yes", "n/a"),
    6:  ("yes", "yes", "n/a"),
    7:  ("yes", "yes", "n/a"),
    8:  ("yes", "yes", "n/a"),
    9:  ("yes", "yes", "n/a"),
    10: ("yes", "yes", "n/a"),
    11: ("yes", "yes", "n/a"),
    12: ("yes", "yes", "n/a"),
    13: ("yes", "yes", "n/a"),
    14: ("n/a", "n/a", "yes"),
    15: ("n/a", "n/a", "yes"),
    16: ("n/a", "n/a", "yes"),
    17: ("n/a", "n/a", "yes"),
    18: ("n/a", "n/a", "yes"),
}

eval_df = pd.read_csv("evaluation/evaluation_results_raw.csv")
eval_df["correctness"] = eval_df["id"].map(lambda x: scores[x][0])
eval_df["grounded"] = eval_df["id"].map(lambda x: scores[x][1])
eval_df["refusal_appropriate"] = eval_df["id"].map(lambda x: scores[x][2])

eval_df.to_csv("evaluation/evaluation_results.csv", index=False)

# Print summary
correct = sum(1 for v in scores.values() if v[0] == "yes")
partial = sum(1 for v in scores.values() if v[0] == "partial")
refusals = sum(1 for v in scores.values() if v[2] == "yes")

print(f"Total questions: 18")
print(f"Correct:         {correct}/13 answered")
print(f"Partial:         {partial}/13 answered")
print(f"Refusals correct: {refusals}/5")
print(f"\nSaved to evaluation/evaluation_results.csv ✓")

Total questions: 18
Correct:         11/13 answered
Partial:         2/13 answered
Refusals correct: 5/5

Saved to evaluation/evaluation_results.csv ✓


## 7. Dense and hybrid retrieval with MiniLM

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load model
print("Loading sentence transformer...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all chunks
print("Embedding chunks...")
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedder.encode(chunk_texts, show_progress_bar=True)

print(f"Embeddings shape: {chunk_embeddings.shape}")

def dense_retrieve(query, top_k=3):
    query_embedding = embedder.encode([query])
    
    # Cosine similarity
    scores = np.dot(chunk_embeddings, query_embedding.T).flatten()
    scores = scores / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-9)
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["dense_score"] = round(float(scores[idx]), 4)
        results.append(chunk)
    return results

# Compare BM25 vs Dense on same 3 queries
test_queries = [
    "What is displacement and how is it different from distance?",
    "State Newton's second law of motion with formula",
    "What happens to velocity in uniform circular motion?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    bm25_results = retrieve(query, top_k=1)
    dense_results = dense_retrieve(query, top_k=1)
    
    print(f"\n  BM25  Rank1 | {bm25_results[0]['content_type']} | Page {bm25_results[0]['page']} | {bm25_results[0]['section']}")
    print(f"  Dense Rank1 | {dense_results[0]['content_type']} | Page {dense_results[0]['page']} | {dense_results[0]['section']}")

Loading sentence transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Ace\OneDrive\Desktop\parishiksha_study_assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ace\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (71, 384)

Query: What is displacement and how is it different from distance?

  BM25  Rank1 | concept | Page 23 | 4.4 Motion in a Plane
  Dense Rank1 | end_of_chapter | Page 21 | 4.4 Motion in a Plane

Query: State Newton's second law of motion with formula

  BM25  Rank1 | activity | Page 11 | 6.5 Newton’s Second Law of Motion
  Dense Rank1 | end_of_chapter | Page 19 | 6.7 Forces Acting on a System of Objects

Query: What happens to velocity in uniform circular motion?

  BM25  Rank1 | concept | Page 19 | 4.4 Motion in a Plane
  Dense Rank1 | concept | Page 19 | 4.4 Motion in a Plane


In [19]:
def hybrid_retrieve(query, top_k=3, alpha=0.5):
    """
    alpha = weight for dense score (1-alpha = weight for BM25)
    """
    # BM25 scores
    query_tokens = simple_tokenize(query)
    bm25_scores = bm25.get_scores(query_tokens)
    
    # Normalize BM25
    bm25_max = max(bm25_scores) + 1e-9
    bm25_norm = bm25_scores / bm25_max
    
    # Dense scores
    query_embedding = embedder.encode([query])
    dense_scores = np.dot(chunk_embeddings, query_embedding.T).flatten()
    dense_scores = dense_scores / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding) + 1e-9)
    
    # Normalize dense
    dense_max = max(dense_scores) + 1e-9
    dense_norm = dense_scores / dense_max
    
    # Combine
    hybrid_scores = alpha * dense_norm + (1 - alpha) * bm25_norm
    
    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        chunk = chunks[idx].copy()
        chunk["hybrid_score"] = round(float(hybrid_scores[idx]), 4)
        results.append(chunk)
    return results

# Compare all three
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    bm25_r = retrieve(query, top_k=1)
    dense_r = dense_retrieve(query, top_k=1)
    hybrid_r = hybrid_retrieve(query, top_k=1)
    
    print(f"  BM25   | Page {bm25_r[0]['page']} | {bm25_r[0]['section'][:45]}")
    print(f"  Dense  | Page {dense_r[0]['page']} | {dense_r[0]['section'][:45]}")
    print(f"  Hybrid | Page {hybrid_r[0]['page']} | {hybrid_r[0]['section'][:45]}")


Query: What is displacement and how is it different from distance?
  BM25   | Page 23 | 4.4 Motion in a Plane
  Dense  | Page 21 | 4.4 Motion in a Plane
  Hybrid | Page 21 | 4.4 Motion in a Plane

Query: State Newton's second law of motion with formula
  BM25   | Page 11 | 6.5 Newton’s Second Law of Motion
  Dense  | Page 19 | 6.7 Forces Acting on a System of Objects
  Hybrid | Page 19 | 6.7 Forces Acting on a System of Objects

Query: What happens to velocity in uniform circular motion?
  BM25   | Page 19 | 4.4 Motion in a Plane
  Dense  | Page 19 | 4.4 Motion in a Plane
  Hybrid | Page 19 | 4.4 Motion in a Plane


## 8. Wk10: token-aware chunking (250 tokens, 40 overlap)

In [21]:
import tiktoken
import json
import pandas as pd
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4",
    chunk_size=250,
    chunk_overlap=40,
)

print("Splitter loaded ✓")
print(f"Test: 'displacement' → {count_tokens('displacement')} tokens")

Splitter loaded ✓
Test: 'displacement' → 2 tokens


In [22]:
import json
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    existing_chunks = json.load(f)

all_sections = [{
    "chapter": c["chapter"],
    "section": c["section"],
    "content_type": c["content_type"],
    "page": c["page"],
    "text": c["text"]
} for c in existing_chunks]

print(f"Loaded {len(all_sections)} sections from existing chunks ✓")

Loaded 71 sections from existing chunks ✓


In [23]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'Chapter-\d+\.indd.*', '', text).strip()
    text = re.sub(r'\d*\s*Exploration\|Grade\s*9\s*\d*', '', text).strip()
    text = re.sub(r'Fig\.?\s*\d+\.\d+[a-z]?\s*:?[^\n]*', '', text).strip()
    return text.strip()

def build_wk10_chunks(sections):
    wk10_chunks = []
    chunk_id = 0
    for section in sections:
        text = clean_text(section["text"])
        ctype = section["content_type"]
        if len(text.split()) < 20:
            continue
        metadata = {
            "source": section["chapter"],
            "section": section["section"],
            "content_type": ctype,
            "page": section["page"]
        }
        if ctype == "worked_example":
            wk10_chunks.append({
                "chunk_id": f"wk10_{chunk_id:04d}",
                **metadata,
                "text": text,
                "token_count": count_tokens(text)
            })
            chunk_id += 1
        elif ctype == "end_of_chapter":
            questions = re.split(r'(?=\n?\d+\.\s+[A-Z])', text)
            for q in questions:
                q = q.strip()
                if len(q.split()) > 15:
                    wk10_chunks.append({
                        "chunk_id": f"wk10_{chunk_id:04d}",
                        **metadata,
                        "text": q,
                        "token_count": count_tokens(q)
                    })
                    chunk_id += 1
        else:
            # LangChain splits here
            split_texts = splitter.split_text(text)
            for st in split_texts:
                if len(st.strip()) > 20:
                    wk10_chunks.append({
                        "chunk_id": f"wk10_{chunk_id:04d}",
                        **metadata,
                        "text": st.strip(),
                        "token_count": count_tokens(st)
                    })
                    chunk_id += 1
    return wk10_chunks

wk10_chunks = build_wk10_chunks(all_sections)

with open("data/processed/wk10_chunks.json", "w", encoding="utf-8") as f:
    json.dump(wk10_chunks, f, indent=2, ensure_ascii=False)

wk10_df = pd.DataFrame(wk10_chunks)
print(f"Total Wk10 chunks: {len(wk10_chunks)}")
print(f"\nBy content type:")
print(wk10_df["content_type"].value_counts())
print(f"\nToken stats:")
print(wk10_df["token_count"].describe().round(1))
print(f"\nChunks over 250 tokens: {(wk10_df['token_count'] > 250).sum()}")

# View sample chunks
for c in wk10_chunks[:5]:
    print(f"{'─'*60}")
    print(f"ID: {c['chunk_id']} | Type: {c['content_type']} | Tokens: {c['token_count']}")
    print(f"Section: {c['section']}")
    print(f"Text: {c['text'][:200]}")

Total Wk10 chunks: 81

By content type:
content_type
activity          35
concept           28
end_of_chapter     9
worked_example     9
Name: count, dtype: int64

Token stats:
count     81.0
mean     148.9
std      111.2
min       36.0
25%       75.0
50%      112.0
75%      196.0
max      594.0
Name: token_count, dtype: float64

Chunks over 250 tokens: 5
────────────────────────────────────────────────────────────
ID: wk10_0000 | Type: concept | Tokens: 246
Section: Introduction
Text: Describing Motion Around Us Chapter 4 Everything in nature is in motion, from massive astronomical objects to subatomic particles. We have a wide variety of motion in nature, such as flitting butterfl
────────────────────────────────────────────────────────────
ID: wk10_0001 | Type: concept | Tokens: 131
Section: 4.1 Motion in a Straight Line
Text: Describing Motion Around Us 49 about some more physical quantities, such as displacement, average velocity and average acceleration. You will also learn to de

In [24]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
  
wk10_docs = [
    Document(
        page_content=c["text"],
        metadata={
            "chunk_id": c["chunk_id"],
            "section": c["section"],
            "content_type": c["content_type"],
            "page": c["page"]
        }
    )
    for c in wk10_chunks
]

retriever = BM25Retriever.from_documents(wk10_docs, k=5)
print(f"BM25 retriever ready: {len(wk10_docs)} docs ✓")

query = "What is displacement?"
results = retriever.invoke(query)

for i, r in enumerate(results, 1):
    print(f"\nRank {i}")
    print(f"  Section: {r.metadata['section']}")
    print(f"  Type: {r.metadata['content_type']}")
    print(f"  Page: {r.metadata['page']}")
    print(f"  Text: {r.page_content[:150]}")

BM25 retriever ready: 81 docs ✓

Rank 1
  Section: 4.4 Motion in a Plane
  Type: end_of_chapter
  Page: 21
  Text: 1. My father went to a shop from home which is located at a distance of 250 m on a straight road. On reaching there, he discovered that he forgot to c

Rank 2
  Section: 4.4 Motion in a Plane
  Type: activity
  Page: 20
  Text: you lift the ring while the marble is moving. 4. Now, after one or two complete revolutions of the marble, pick up the ring without disturbing the mot

Rank 3
  Section: 4.2.3 Velocity-time graphs
  Type: concept
  Page: 15
  Text: Which physical quantities can be obtained from a velocity-time graph? Apart from finding the magnitude of velocity of the object at each instant of ti

Rank 4
  Section: 4.1.4 Average acceleration
  Type: concept
  Page: 7
  Text: The average velocity of an object over a large time interval may differ from its velocity at a particular instant. Throughout this chapter, we will us

Rank 5
  Section: 4.2.2 Position-time grap

## 9. BM25 on Wk9 vs Wk10 chunks

In [25]:
# Load Wk9 chunks
with open("data/processed/chunks.json") as f:
    wk9_chunks = json.load(f)

# Rebuild the Wk9 BM25 retriever for comparison
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

wk9_corpus = [tokenize(c["text"]) for c in wk9_chunks]
wk9_bm25 = BM25Okapi(wk9_corpus)

def retrieve_wk9(query, top_k=3):
    tokens = tokenize(query)
    scores = wk9_bm25.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    results = []
    for idx in top_indices:
        c = wk9_chunks[idx].copy()
        c["bm25_score"] = round(scores[idx], 4)
        results.append(c)
    return results

# 5 comparison queries
test_queries = [
    "What is displacement?",
    "State Newton's second law of motion",
    "What is uniform circular motion?",
    "How does friction affect motion?",
    "What is the difference between speed and velocity?"
]

comparison_log = []

for query in test_queries:
    print(f"\n{'='*65}")
    print(f"Query: {query}")
    print(f"{'='*65}")

    wk9_results = retrieve_wk9(query, top_k=3)
    wk10_results = retriever.invoke(query)[:3]

    print(f"\n{'WK9 (word-based, 300w)':^30} | {'WK10 (tiktoken, 250tok)':^30}")
    print("-" * 65)

    for i in range(3):
        w9 = wk9_results[i] if i < len(wk9_results) else None
        w10 = wk10_results[i] if i < len(wk10_results) else None

        w9_label = f"[{w9['content_type']}] {w9['section'][:25]}" if w9 else "—"
        w10_label = f"[{w10.metadata['content_type']}] {w10.metadata['section'][:25]}" if w10 else "—"

        print(f"R{i+1}: {w9_label:<30} | {w10_label:<30}")

    comparison_log.append({
        "query": query,
        "wk9_top3": [{"chunk_id": r["chunk_id"], "section": r["section"], 
                      "content_type": r["content_type"], "score": r["bm25_score"]} 
                     for r in wk9_results],
        "wk10_top3": [{"chunk_id": r.metadata["chunk_id"], "section": r.metadata["section"],
                       "content_type": r.metadata["content_type"]} 
                      for r in wk10_results]
    })

# Save log
with open("data/processed/bm25_comparison_log.json", "w") as f:
    json.dump(comparison_log, f, indent=2)

print("\n\nLog saved to bm25_comparison_log.json ✓")


Query: What is displacement?

    WK9 (word-based, 300w)     |    WK10 (tiktoken, 250tok)    
-----------------------------------------------------------------
R1: [concept] 6.1 The Concept of Force | [end_of_chapter] 4.4 Motion in a Plane
R2: [concept] 4.4 Motion in a Plane | [activity] 4.4 Motion in a Plane
R3: [concept] 4.2.3 Velocity-time graph | [concept] 4.2.3 Velocity-time graph

Query: State Newton's second law of motion

    WK9 (word-based, 300w)     |    WK10 (tiktoken, 250tok)    
-----------------------------------------------------------------
R1: [end_of_chapter] 6.7 Forces Acting on a Sy | [activity] 6.5 Newton’s Second Law o
R2: [activity] 6.5 Newton’s Second Law o | [activity] 6.5 Newton’s Second Law o
R3: [activity] 6.4 Newton’s First Law of | [end_of_chapter] 6.7 Forces Acting on a Sy

Query: What is uniform circular motion?

    WK9 (word-based, 300w)     |    WK10 (tiktoken, 250tok)    
-----------------------------------------------------------------
R1: [concep

## 10. Dense index: Gemini embeddings in Chroma

In [41]:
import json, os, time
from dotenv import load_dotenv
import chromadb
from google import genai

load_dotenv()

client_gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

CHROMA_PATH     = "./chroma_wk10"
COLLECTION_NAME = "parishiksha_wk10"
EMBED_MODEL     = "gemini-embedding-001"  

# Load chunks
with open("data/processed/wk10_chunks.json", encoding="utf-8") as f:
    wk10_chunks = json.load(f)

client_chroma = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client_chroma.delete_collection(COLLECTION_NAME)
    print("Deleted old collection.")
except Exception:
    pass  # collection did not exist yet

collection = client_chroma.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

def embed_texts(texts: list) -> list:
    """Embed a list of texts using gemini-embedding-001."""
    embeddings = []
    for text in texts:
        result = client_gemini.models.embed_content(
            model=EMBED_MODEL,
            contents=text
        )
        embeddings.append(result.embeddings[0].values)
        time.sleep(0.1)  
    return embeddings

# Embed and index
print(f"Embedding {len(wk10_chunks)} chunks with {EMBED_MODEL}...")
batch_size = 20
total_batches = -(-len(wk10_chunks) // batch_size)

for i in range(0, len(wk10_chunks), batch_size):
    batch = wk10_chunks[i : i + batch_size]
    texts = [c["text"] for c in batch]
    vectors = embed_texts(texts)

    collection.add(
        ids        = [c["chunk_id"] for c in batch],
        embeddings = vectors,
        documents  = texts,
        metadatas  = [{
            "chunk_id":     c["chunk_id"],
            "source":       c["source"],
            "section":      c["section"],
            "content_type": c["content_type"],
            "page":         c["page"],
            "token_count":  c["token_count"]
        } for c in batch]
    )
    print(f"  Batch {i//batch_size + 1}/{total_batches} — collection size: {collection.count()}")

print(f"\nDone. Final collection count: {collection.count()}")

Deleted old collection.
Embedding 81 chunks with gemini-embedding-001...
  Batch 1/5 — collection size: 20
  Batch 2/5 — collection size: 40
  Batch 3/5 — collection size: 60
  Batch 4/5 — collection size: 80
  Batch 5/5 — collection size: 81

Done. Final collection count: 81


In [42]:
# IMPORTANT: query-side must use the same EMBED_MODEL as index-side (gemini-embedding-001)

def embed_query(text: str) -> list:
    """Embed a single query using the same model used at index time."""
    result = client_gemini.models.embed_content(
        model=EMBED_MODEL,
        contents=text
    )
    return result.embeddings[0].values

def retrieve(query: str, k: int = 5) -> list:
    """Return top-k chunks with similarity scores from Chroma."""
    query_vector = embed_query(query)
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        # Chroma cosine distance → similarity = 1 - distance
        chunks.append({
            "chunk_id":     meta["chunk_id"],
            "section":      meta["section"],
            "content_type": meta["content_type"],
            "page":         meta["page"],
            "similarity":   round(1 - dist, 4),
            "text":         doc
        })
    return chunks

# ── Smoke test ──
print("Smoke test — 'What is displacement?'\n")
test_results = retrieve("What is displacement?", k=5)
for r in test_results:
    print(f"  [{r['similarity']:.3f}] {r['chunk_id']} | {r['section'][:55]}")
    print(f"           {r['text'][:120]}")
    print()

Smoke test — 'What is displacement?'

  [0.632] wk10_0023 | 4.3 Kinematic Equations for Motion in a Straight
           Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the

  [0.615] wk10_0033 | 4.4 Motion in a Plane
           2. A student runs from the ground floor to the fourth floor of a school building to collect a book and then comes down t

  [0.614] wk10_0032 | 4.4 Motion in a Plane
           1. My father went to a shop from home which is located at a distance of 250 m on a straight road. On reaching there, he 

  [0.590] wk10_0001 | 4.1 Motion in a Straight Line
           Describing Motion Around Us 49 about some more physical quantities, such as displacement, average velocity and average a

  [0.582] wk10_0015 | 4.2.2 Position-time graphs
           equal (20 m) magnitude of velocity is constant In equal intervals of time (say, 4 – 6 s and 10 – 12 s) the magnitudes of



## 11. Retrieval log: top-1 hit rate on 10 questions

In [ ]:
import json
from pathlib import Path

eval_questions = [
    {"id": 1,  "question": "What is displacement?"},
    {"id": 2,  "question": "What are the three kinematic equations of motion?"},
    {"id": 3,  "question": "What is uniform circular motion?"},
    {"id": 4,  "question": "What is average acceleration?"},
    {"id": 5,  "question": "State Newton's first law of motion."},
    {"id": 6,  "question": "State Newton's second law of motion."},
    {"id": 7,  "question": "State Newton's third law of motion."},
    {"id": 8,  "question": "What is the SI unit of force and how is it defined?"},
    {"id": 9,  "question": "What is the force of friction and in which direction does it act?"},
    {"id": 10, "question": "How do you calculate average speed?"},
]

retrieval_log = []

print(f"{'Q':>2}  {'Sim':>5}  {'chunk_id':<14}  {'Section':<45}")
print("─" * 80)

for q in eval_questions:
    results = retrieve(q["question"], k=5)
    top1 = results[0]

    print(f"\nQ{q['id']:>2}  [{top1['similarity']:.3f}]  {top1['chunk_id']:<14}  {top1['section'][:45]}")
    print(f"      Question : {q['question']}")
    print(f"      Preview  : {top1['text'][:200]}")
    print(f"      → Answer present? (judge manually after reading preview)")

    retrieval_log.append({
        "id":               q["id"],
        "question":         q["question"],
        "top1_chunk_id":    top1["chunk_id"],
        "top1_section":     top1["section"],
        "top1_similarity":  top1["similarity"],
        "top1_preview":     top1["text"][:300],
        "answer_present":   "FILL_IN",  # you will update this in next cell
        "top5": [
            {"chunk_id": r["chunk_id"], "section": r["section"], "similarity": r["similarity"]}
            for r in results
        ]
    })

Path("data/processed").mkdir(parents=True, exist_ok=True)
with open("data/processed/retrieval_log_draft.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_log, f, indent=2)

print("\n\nDraft saved to retrieval_log_draft.json")
print("Now read the previews above and fill in YES/NO in the next cell.")

 Q    Sim  chunk_id        Section                                      
────────────────────────────────────────────────────────────────────────────────

Q 1  [0.632]  wk10_0023       4.3 Kinematic Equations for Motion in a Strai
      Question : What is displacement?
      Preview  : Describing Motion Around Us 63 This is the displacement of the car from the origin in 6 seconds. You have found that the area enclosed by the velocity-time graph and the time axis for a desired time i
      → Answer present? (judge manually after reading preview)

Q 2  [0.720]  wk10_0026       4.3 Kinematic Equations for Motion in a Strai
      Question : What are the three kinematic equations of motion?
      Preview  : As learnt in the earlier section, the displacement s of the object during the time interval t is given by the area enclosed within OABD. Thus, s = area of OABD = area of the rectangle OACD + area of t
      → Answer present? (judge manually after reading preview)

Q 3  [0.718]  wk10_0029

In [ ]:
import json

judgements = {
    1:  "NO",   # definition of displacement not in top-1 chunk
    2:  "NO",   # derivation shown but 3 final equations not clearly listed
    3:  "YES",  # uniform circular motion defined in preview
    4:  "YES",  # average acceleration defined in preview
    5:  "NO",   # inertia discussed but law not stated in top-1
    6:  "YES",  # Newton's second law stated clearly
    7:  "YES",  # Newton's third law — equal and opposite force present
    8:  "YES",  # SI unit of force, F=ma present
    9:  "YES",  # friction direction stated clearly
    10: "YES",  # average speed section present
}

with open("data/processed/retrieval_log_draft.json", encoding="utf-8") as f:
    retrieval_log = json.load(f)

for entry in retrieval_log:
    entry["answer_present"] = judgements[entry["id"]]

with open("data/processed/retrieval_log.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_log, f, indent=2)

yes_count = sum(1 for e in retrieval_log if e["answer_present"] == "YES")
no_count  = sum(1 for e in retrieval_log if e["answer_present"] == "NO")

print(f"retrieval_log.json saved ✓")
print(f"Top-1 hit rate: {yes_count}/10")
print(f"Misses (NO):    {no_count}/10")
print()
for e in retrieval_log:
    print(f"  Q{e['id']:>2}  {e['answer_present']}  [{e['top1_similarity']:.3f}]  {e['top1_section'][:50]}")

retrieval_log.json saved ✓
Top-1 hit rate: 7/10
Misses (NO):    3/10

  Q 1  NO  [0.632]  4.3 Kinematic Equations for Motion in a Straight
  Q 2  NO  [0.720]  4.3 Kinematic Equations for Motion in a Straight
  Q 3  YES  [0.718]  4.4 Motion in a Plane
  Q 4  YES  [0.712]  4.1.4 Average acceleration
  Q 5  NO  [0.728]  6.4 Newton’s First Law of Motion
  Q 6  YES  [0.756]  6.5 Newton’s Second Law of Motion
  Q 7  YES  [0.731]  6.6 Newton’s Third Law of Motion
  Q 8  YES  [0.687]  6.5 Newton’s Second Law of Motion
  Q 9  YES  [0.716]  6.7 Forces Acting on a System of Objects
  Q10  YES  [0.690]  4.1.3 Average speed and average velocity


## 12. Prompt design: permissive vs strict
Write-up: [`docs/prompt_design.md`](../docs/prompt_design.md)

In [46]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "llama-3.3-70b-versatile"

def format_context(chunks: list) -> str:
    parts = []
    for c in chunks:
        parts.append(f"[{c['chunk_id']} | {c['section']}]\n{c['text']}")
    return "\n\n---\n\n".join(parts)

PERMISSIVE_PROMPT = """You are a study assistant for Class 9 Science students.
Answer the student's question using the context below.

Context:
{context}

Question: {question}
Answer:"""

def ask_permissive(question: str, k: int = 5) -> str:
    chunks = retrieve(question, k=k)
    context = format_context(chunks)
    prompt = PERMISSIVE_PROMPT.format(context=context, question=question)
    response = groq_client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content.strip()

# 3 test queries — 2 in-scope + 1 out-of-scope
test_cases = [
    "What is Newton's second law of motion?",
    "How do you calculate average speed?",
    "What is the speed of light in a vacuum?",  # out-of-scope — watch for hallucination
]

permissive_results = {}
for q in test_cases:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    answer = ask_permissive(q)
    permissive_results[q] = answer
    print(f"A (permissive):\n{answer}")


Q: What is Newton's second law of motion?
A (permissive):
Newton's second law of motion states that when a net force acts on an object, the object accelerates in the direction of the net force. The magnitude of the acceleration is proportional to the magnitude of the net force and is inversely proportional to the mass of the object. This means that for a given object, a larger force will result in a larger acceleration, while a smaller force will result in a smaller acceleration. Similarly, for a given force, an object with a smaller mass will accelerate more than an object with a larger mass.

Q: How do you calculate average speed?
A (permissive):
To calculate average speed, you divide the total distance travelled by the time interval during which this distance is covered. The formula for average speed is:

Average speed = Total distance travelled / Time interval

This can be represented mathematically as:

Average speed = Distance / Time 

For example, if an object travels a total d

In [48]:
import re

STRICT_PROMPT = """You are a study assistant for Class 9 Science students at PariShiksha.
Answer ONLY using the context provided below. Do NOT use any outside knowledge whatsoever.
If the answer is not present in the context, reply EXACTLY with this phrase and nothing else:
I don't have that in my study materials.

After every factual claim you MUST cite the source chunk using EXACTLY this format: [Source: wk10_XXXX]
Where wk10_XXXX is the chunk_id from the context header. For example: [Source: wk10_0042]
Do not use any other citation format. Do not combine multiple chunk_ids in one bracket.

Context:
{context}

Question: {question}
Answer:"""

def ask(question: str, k: int = 5) -> dict:
    """Returns dict with answer, sources, chunk_ids."""
    chunks = retrieve(question, k=k)

    # Always print retrieved chunks before calling model
    print(f"  [Retrieved chunks for: '{question[:55]}']")
    for c in chunks:
        print(f"    [{c['similarity']:.3f}] {c['chunk_id']} | {c['section'][:50]}")

    context = format_context(chunks)
    prompt = STRICT_PROMPT.format(context=context, question=question)

    response = groq_client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    answer_text = response.choices[0].message.content.strip()

    # Match exactly [Source: wk10_XXXX]
    cited_ids = re.findall(r'\[Source:\s*(wk10_\d+)\]', answer_text)

    return {
        "answer":           answer_text,
        "sources":          [c["section"] for c in chunks if c["chunk_id"] in cited_ids],
        "chunk_ids":        cited_ids,
        "retrieved_chunks": [{"chunk_id": c["chunk_id"], "section": c["section"],
                              "similarity": c["similarity"]} for c in chunks]
    }

# Re-run same 3 queries
strict_results = {}
for q in test_cases:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    result = ask(q)
    strict_results[q] = result
    print(f"A (strict):\n{result['answer']}")
    print(f"Cited chunk_ids: {result['chunk_ids']}")


Q: What is Newton's second law of motion?
  [Retrieved chunks for: 'What is Newton's second law of motion?']
    [0.732] wk10_0058 | 6.5 Newton’s Second Law of Motion
    [0.703] wk10_0051 | 6.5 Newton’s Second Law of Motion
    [0.675] wk10_0061 | 6.5 Newton’s Second Law of Motion
    [0.670] wk10_0074 | 6.7 Forces Acting on a System of Objects
    [0.670] wk10_0055 | 6.5 Newton’s Second Law of Motion
A (strict):
Newton's second law of motion states that when a net force acts on an object, the object accelerates in the direction of the net force. The magnitude of the acceleration is proportional to the magnitude of the net force and is inversely proportional to the mass of the object. [Source: wk10_0058]
Cited chunk_ids: ['wk10_0058']

Q: How do you calculate average speed?
  [Retrieved chunks for: 'How do you calculate average speed?']
    [0.690] wk10_0005 | 4.1.3 Average speed and average velocity
    [0.647] wk10_0006 | 4.1.3 Average speed and average velocity
    [0.641] wk10_00

## 13. Wk10 evaluation: 12 questions on 3 axes

In [50]:
# 6 direct + 3 paraphrased + 3 OOS (last one is plausibly answerable)
import pandas as pd
from pathlib import Path

eval_set = [
    # Direct (6)
    {"id": 1,  "category": "direct",       "question": "What is displacement?"},
    {"id": 2,  "category": "direct",       "question": "What are the three kinematic equations of motion?"},
    {"id": 3,  "category": "direct",       "question": "What is uniform circular motion?"},
    {"id": 4,  "category": "direct",       "question": "What is average acceleration?"},
    {"id": 5,  "category": "direct",       "question": "State Newton's first law of motion."},
    {"id": 6,  "category": "direct",       "question": "What is the SI unit of force and how is it defined?"},
    # Paraphrased (3)
    {"id": 7,  "category": "paraphrased",  "question": "Why does a fielder pull hands back while catching a fast cricket ball?"},
    {"id": 8,  "category": "paraphrased",  "question": "If an object moves in a circle at constant speed, is it accelerating?"},
    {"id": 9,  "category": "paraphrased",  "question": "What decides how far a car travels after brakes are applied?"},
    # Out-of-scope (3)
    {"id": 10, "category": "out_of_scope", "question": "What is photosynthesis?"},
    {"id": 11, "category": "out_of_scope", "question": "Explain the water cycle."},
    {"id": 12, "category": "out_of_scope", "question": "Calculate the value of g on the surface of the Moon."},
]

Path("evaluation").mkdir(exist_ok=True)

raw_rows = []
for item in eval_set:
    print(f"\n{'='*60}")
    print(f"Q{item['id']} [{item['category']}]: {item['question']}")
    result = ask(item["question"])
    top1 = result["retrieved_chunks"][0] if result["retrieved_chunks"] else {}
    raw_rows.append({
        "id":              item["id"],
        "category":        item["category"],
        "question":        item["question"],
        "answer":          result["answer"],
        "chunk_ids_cited": ", ".join(result["chunk_ids"]),
        "top1_chunk_id":   top1.get("chunk_id", ""),
        "top1_section":    top1.get("section", ""),
        "top1_similarity": top1.get("similarity", ""),
        "all_retrieved":   str([r["chunk_id"] for r in result["retrieved_chunks"]])
    })
    print(f"A: {result['answer'][:200]}")
    print(f"Cited: {result['chunk_ids']}")

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv("evaluation/eval_raw.csv", index=False)
print("\n\neval_raw.csv saved ✓")
print(f"Total questions run: {len(raw_rows)}")


Q1 [direct]: What is displacement?
  [Retrieved chunks for: 'What is displacement?']
    [0.632] wk10_0023 | 4.3 Kinematic Equations for Motion in a Straight
    [0.615] wk10_0033 | 4.4 Motion in a Plane
    [0.614] wk10_0032 | 4.4 Motion in a Plane
    [0.590] wk10_0001 | 4.1 Motion in a Straight Line
    [0.582] wk10_0015 | 4.2.2 Position-time graphs
A: Displacement is the shortest distance between the initial and final position of an object. [Source: wk10_0023]
Cited: ['wk10_0023']

Q2 [direct]: What are the three kinematic equations of motion?
  [Retrieved chunks for: 'What are the three kinematic equations of motion?']
    [0.720] wk10_0026 | 4.3 Kinematic Equations for Motion in a Straight
    [0.705] wk10_0024 | 4.3 Kinematic Equations for Motion in a Straight
    [0.672] wk10_0027 | 4.3 Kinematic Equations for Motion in a Straight
    [0.663] wk10_0031 | 4.4 Motion in a Plane
    [0.632] wk10_0074 | 6.7 Forces Acting on a System of Objects
A: The three kinematic equations of m

In [51]:
# correctness:      yes | partial | no     (n/a for OOS)
# grounded:         yes | no               (n/a for OOS)
# refused_when_oos: yes | no               (n/a for non-OOS)

scores = {
    #  id : (correctness, grounded, refused_when_oos)
    1:  ("yes",     "yes", "n/a"),  # displacement defined correctly
    2:  ("yes",     "yes", "n/a"),  # all 3 equations present
    3:  ("yes",     "yes", "n/a"),  # uniform circular motion correct
    4:  ("yes",     "yes", "n/a"),  # average acceleration correct
    5:  ("yes",     "yes", "n/a"),  # Newton's first law stated correctly
    6:  ("yes",     "yes", "n/a"),  # SI unit of force correct
    7:  ("partial", "yes", "n/a"),  # correct but misses the force-time explanation
    8:  ("yes",     "yes", "n/a"),  # circular motion + acceleration correct
    9:  ("yes",     "yes", "n/a"),  # braking distance factors correct
    10: ("n/a",     "n/a", "yes"),  # photosynthesis refused correctly
    11: ("n/a",     "n/a", "yes"),  # water cycle refused correctly
    12: ("n/a",     "n/a", "yes"),  # Moon gravity refused correctly (plausibly answerable)
}

raw_df = pd.read_csv("evaluation/eval_raw.csv")
raw_df["correctness"]      = raw_df["id"].map(lambda x: scores[x][0])
raw_df["grounded"]         = raw_df["id"].map(lambda x: scores[x][1])
raw_df["refused_when_oos"] = raw_df["id"].map(lambda x: scores[x][2])

raw_df.to_csv("evaluation/eval_scored.csv", index=False)

# Summary
answered = raw_df[raw_df["category"] != "out_of_scope"]
oos      = raw_df[raw_df["category"] == "out_of_scope"]

print("=== Eval v1 Summary ===")
print(f"Correct:     {(answered['correctness']=='yes').sum()}/{len(answered)}")
print(f"Partial:     {(answered['correctness']=='partial').sum()}/{len(answered)}")
print(f"Grounded:    {(answered['grounded']=='yes').sum()}/{len(answered)}")
print(f"OOS refused: {(oos['refused_when_oos']=='yes').sum()}/{len(oos)}")

print("""
=== Worst question diagnosis ===
Q7 — "Why does a fielder pull hands back while catching a fast cricket ball?"

Correctness: partial
The answer correctly says pulling back minimises injury but misses the
core physics explanation: pulling back increases the time of contact,
which by Newton's second law (F = Δp/Δt) reduces the force experienced.
The retrieved chunk (wk10_0063) contains the worked example but the
model only extracted the surface-level conclusion, not the force-time
reasoning that makes it a Newton's second law application.

Failure-mode catalog: mixed_structure — the worked example chunk contains
both the setup and the conclusion but the force-time reasoning sits in a
neighbouring chunk that ranked outside top-1. The model answered from
what it was given rather than the complete explanation.
""")

print("eval_scored.csv saved ✓")

=== Eval v1 Summary ===
Correct:     8/9
Partial:     1/9
Grounded:    9/9
OOS refused: 3/3

=== Worst question diagnosis ===
Q7 — "Why does a fielder pull hands back while catching a fast cricket ball?"

Correctness: partial
The answer correctly says pulling back minimises injury but misses the
core physics explanation: pulling back increases the time of contact,
which by Newton's second law (F = Δp/Δt) reduces the force experienced.
The retrieved chunk (wk10_0063) contains the worked example but the
model only extracted the surface-level conclusion, not the force-time
reasoning that makes it a Newton's second law application.

Failure-mode catalog: mixed_structure — the worked example chunk contains
both the setup and the conclusion but the force-time reasoning sits in a
neighbouring chunk that ranked outside top-1. The model answered from
what it was given rather than the complete explanation.

eval_scored.csv saved ✓


## 14. Fix attempt: junk-chunk filter
Write-up: [`docs/fix_memo.md`](../docs/fix_memo.md)

In [58]:
# Check how many chunks get removed
before_count = len(chunks)
chunks_v2    = [c for c in chunks if len(c["text"].split()) > 50]
after_count  = len(chunks_v2)

print(f"Chunks before filter: {before_count}")
print(f"Chunks after  filter: {after_count}")
print(f"Removed: {before_count - after_count}")

# Show what got removed
removed = [c for c in chunks if len(c["text"].split()) <= 50]
print("\n── Removed chunks ──")
for c in removed:
    print(f"  ID:{c['chunk_id']} | words:{len(c['text'].split())} | {c['text'][:100]}")

Chunks before filter: 71
Chunks after  filter: 59
Removed: 12

── Removed chunks ──
  ID:13 | words:36 | 6. Is its motion, a straight line motion? Assuming the starting point of the ball (O) to be the orig
  ID:23 | words:43 | this point. Then, draw a line parallel to the x-axis from the point corresponding to distance 20 m o
  ID:31 | words:32 | and 6 s = area of OABC = 20 m s–1 × 6 s = 120 m Velocity (m s–1) 2.5 5.0 7.5 10.0 12.5 15.0 5 10 15 
  ID:34 | words:31 | is a straight line indicating that the velocity changes with constant acceleration. The slope of the
  ID:37 | words:36 | 4.4 Motion in a Plane Motion in a plane, such as a vehicle overtaking another, the path of a kicked 
  ID:47 | words:44 | 11. A student said, “The Earth moves around the Sun”. In this context, discuss whether an object kep
  ID:52 | words:39 | the effect of the force also changes. Note 6.1.1 Measuring the magnitude of a force How can we measu
  ID:63 | words:39 | Is the reading largest for which the dist

In [62]:
# Root cause diagnosis (corrected): Q7 is a corpus gap, not a retrieval bug.
# The worked_example chunk is intentionally short. Word filter must not touch it.
# Fix: filter junk only from concept/activity/end_of_chapter types.

def ask_v2(question: str, k: int = 5) -> dict:
    raw_chunks = retrieve(question, k=k)
    # Only drop short non-example chunks (junk artifacts like chunk 34)
    chunks_filtered = [
        c for c in raw_chunks
        if c["content_type"] == "worked_example" or len(c["text"].split()) > 50
    ][:5]

    print(f"  [Retrieved {len(raw_chunks)} → kept {len(chunks_filtered)} after filter]")
    for c in chunks_filtered:
        print(f"    [{c['similarity']:.3f}] {c['chunk_id']} | {c['section'][:48]}")

    context  = format_context(chunks_filtered)
    prompt   = STRICT_PROMPT.format(context=context, question=question)
    response = groq_client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    answer_text = response.choices[0].message.content.strip()
    cited_ids   = re.findall(r'\[Source:\s*(wk10_\d+)\]', answer_text)
    return {
        "answer":           answer_text,
        "sources":          [c["section"] for c in chunks_filtered if c["chunk_id"] in cited_ids],
        "chunk_ids":        cited_ids,
        "retrieved_chunks": [{"chunk_id": c["chunk_id"], "section": c["section"],
                              "similarity": c["similarity"]} for c in chunks_filtered]
    }

print("Q7 check:")
r = ask_v2("Why does a fielder pull hands back while catching a fast cricket ball?")
print(r["answer"])

Q7 check:
  [Retrieved 5 → kept 4 after filter]
    [0.680] wk10_0063 | 6.5 Newton’s Second Law of Motion
    [0.622] wk10_0062 | 6.5 Newton’s Second Law of Motion
    [0.590] wk10_0038 | 6.1 The Concept of Force
    [0.590] wk10_0075 | 6.7 Forces Acting on a System of Objects
A fielder pulls their hands back while catching a fast cricket ball to minimize the impact and injury, as applying a smaller force to the moving ball slows it down, thus requiring a smaller force to be applied by the fielder to stop the ball [Source: wk10_0063].


## 15. Re-evaluation after the fix

In [63]:
import pandas as pd
from pathlib import Path

eval_set = [
    {"id": 1,  "category": "direct",       "question": "What is displacement?"},
    {"id": 2,  "category": "direct",       "question": "What are the three kinematic equations of motion?"},
    {"id": 3,  "category": "direct",       "question": "What is uniform circular motion?"},
    {"id": 4,  "category": "direct",       "question": "What is average acceleration?"},
    {"id": 5,  "category": "direct",       "question": "State Newton's first law of motion."},
    {"id": 6,  "category": "direct",       "question": "What is the SI unit of force and how is it defined?"},
    {"id": 7,  "category": "paraphrased",  "question": "Why does a fielder pull hands back while catching a fast cricket ball?"},
    {"id": 8,  "category": "paraphrased",  "question": "If an object moves in a circle at constant speed, is it accelerating?"},
    {"id": 9,  "category": "paraphrased",  "question": "What decides how far a car travels after brakes are applied?"},
    {"id": 10, "category": "out_of_scope", "question": "What is photosynthesis?"},
    {"id": 11, "category": "out_of_scope", "question": "Explain the water cycle."},
    {"id": 12, "category": "out_of_scope", "question": "Calculate the value of g on the surface of the Moon."},
]

Path("evaluation").mkdir(exist_ok=True)
raw_v2_rows = []

for item in eval_set:
    print(f"\n{'='*60}")
    print(f"Q{item['id']} [{item['category']}]: {item['question']}")
    result = ask_v2(item["question"])
    top1   = result["retrieved_chunks"][0] if result["retrieved_chunks"] else {}
    raw_v2_rows.append({
        "id":              item["id"],
        "category":        item["category"],
        "question":        item["question"],
        "answer":          result["answer"],
        "chunk_ids_cited": ", ".join(result["chunk_ids"]),
        "top1_chunk_id":   top1.get("chunk_id", ""),
        "top1_section":    top1.get("section", ""),
        "top1_similarity": top1.get("similarity", ""),
        "all_retrieved":   str([r["chunk_id"] for r in result["retrieved_chunks"]])
    })
    print(f"A: {result['answer'][:200]}")
    print(f"Cited: {result['chunk_ids']}")

raw_v2_df = pd.DataFrame(raw_v2_rows)
raw_v2_df.to_csv("evaluation/eval_v2_raw.csv", index=False)
print("\neval_v2_raw.csv saved ✓")


Q1 [direct]: What is displacement?
  [Retrieved 5 → kept 4 after filter]
    [0.632] wk10_0023 | 4.3 Kinematic Equations for Motion in a Straight
    [0.615] wk10_0033 | 4.4 Motion in a Plane
    [0.614] wk10_0032 | 4.4 Motion in a Plane
    [0.590] wk10_0001 | 4.1 Motion in a Straight Line
A: I don't have that in my study materials.
Cited: []

Q2 [direct]: What are the three kinematic equations of motion?
  [Retrieved 5 → kept 5 after filter]
    [0.720] wk10_0026 | 4.3 Kinematic Equations for Motion in a Straight
    [0.705] wk10_0024 | 4.3 Kinematic Equations for Motion in a Straight
    [0.672] wk10_0027 | 4.3 Kinematic Equations for Motion in a Straight
    [0.663] wk10_0031 | 4.4 Motion in a Plane
    [0.632] wk10_0074 | 6.7 Forces Acting on a System of Objects
A: The three kinematic equations of motion are: 
v = u + at, 
s = ut + (1/2)at^2, 
v^2 = u^2 + 2as [Source: wk10_0026]
Cited: ['wk10_0026']

Q3 [direct]: What is uniform circular motion?
  [Retrieved 5 → kept 4 after filt

In [65]:
scores_v2 = {
    #  id : (correctness, grounded, refused_when_oos)
    1:  ("no",      "no",  "n/a"),  # REGRESSION: refused when answer exists in corpus
    2:  ("yes",     "yes", "n/a"),
    3:  ("yes",     "yes", "n/a"),
    4:  ("yes",     "yes", "n/a"),
    5:  ("yes",     "yes", "n/a"),
    6:  ("yes",     "yes", "n/a"),
    7:  ("partial", "yes", "n/a"),  # unchanged — corpus gap
    8:  ("yes",     "yes", "n/a"),
    9:  ("yes",     "yes", "n/a"),
    10: ("n/a",     "n/a", "yes"),
    11: ("n/a",     "n/a", "yes"),
    12: ("n/a",     "n/a", "yes"),
}

import pandas as pd
raw_v2_df = pd.read_csv("evaluation/eval_v2_raw.csv")
raw_v2_df["correctness"]      = raw_v2_df["id"].map(lambda x: scores_v2[x][0])
raw_v2_df["grounded"]         = raw_v2_df["id"].map(lambda x: scores_v2[x][1])
raw_v2_df["refused_when_oos"] = raw_v2_df["id"].map(lambda x: scores_v2[x][2])
raw_v2_df.to_csv("evaluation/eval_v2_scored.csv", index=False)

answered_v2 = raw_v2_df[raw_v2_df["category"] != "out_of_scope"]
oos_v2      = raw_v2_df[raw_v2_df["category"] == "out_of_scope"]

print("=== Eval v1 → v2 Delta ===")
print(f"Correct:     8/9 → {(answered_v2['correctness']=='yes').sum()}/9")
print(f"Partial:     1/9 → {(answered_v2['correctness']=='partial').sum()}/9")
print(f"No/refused:  0/9 → {(answered_v2['correctness']=='no').sum()}/9")
print(f"Grounded:    9/9 → {(answered_v2['grounded']=='yes').sum()}/9")
print(f"OOS refused: 3/3 → {(oos_v2['refused_when_oos']=='yes').sum()}/3")
print()
print("Q1 regressed: answer existed in corpus, filter removed rank-5 chunk,")
print("model over-refused. Fix caused net -1 correct, -1 grounded.")
print()
print("eval_v2_scored.csv saved ✓")

=== Eval v1 → v2 Delta ===
Correct:     8/9 → 7/9
Partial:     1/9 → 1/9
No/refused:  0/9 → 1/9
Grounded:    9/9 → 8/9
OOS refused: 3/3 → 3/3

Q1 regressed: answer existed in corpus, filter removed rank-5 chunk,
model over-refused. Fix caused net -1 correct, -1 grounded.

eval_v2_scored.csv saved ✓
